In [1]:
import xarray as xr
from dask.distributed import Client
from dask_jobqueue import PBSCluster
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
cluster = PBSCluster(
    cores=1,
    memory='50GB',
    processes=1,
    queue='casper',
    local_directory='$TMPDIR',
    account='P93300313',
    walltime='4:00:00'
)
cluster.scale(jobs=8)
client = Client(cluster)
client

In [50]:
path = '/glade/work/acruz/Caribbean_Heat_data/ERA5/'
hi_ds = xr.open_dataset(path+'hourly_HI.nc')
MVIMD_ds  = xr.open_zarr(path+'MVIMD')
MSDRSWRF_ds  = xr.open_zarr(path+'MSDRSWRF')

In [59]:
''' 
This makes an additional dimension which I can select with datetime64
while referenceing the Multi Index pointers.
'''
MVIMD_stack = MVIMD_ds.stack(time=('forecast_initial_time', 'forecast_hour'))
MSDRSWRF_stack = MSDRSWRF_ds.stack(time=('forecast_initial_time', 'forecast_hour'))

# both variables have the same forecast times, -1 to account for forecast_initial_time
actual_time = MVIMD_stack['forecast_initial_time'] + (MVIMD_stack['forecast_hour'] - 1).astype('timedelta64[h]')

MVIMD_fxtime = MVIMD_stack.assign_coords(fixed_time=actual_time).sortby('time').drop_duplicates('time').swap_dims({'time': 'fixed_time'})
MSDRSWRF_fxtime = MSDRSWRF_stack.assign_coords(fixed_time=actual_time).sortby('time').drop_duplicates('time').swap_dims({'time': 'fixed_time'})

In [60]:
dates = slice('1981', '2025')
MVIMD_fxtime = MVIMD_fxtime['MVIMD'].sel(fixed_time=dates)
MSDRSWRF_fxtime = MSDRSWRF_fxtime['MSDRSWRF'].sel(fixed_time=dates)
hi_ds = hi_ds.sel(time=dates).chunk(time=17166)

In [61]:
MVIMD_fxtime

<xarray.DataArray 'MVIMD' (latitude: 82, longitude: 121, fixed_time: 394458)> Size: 16GB
dask.array<getitem, shape=(82, 121, 394458), dtype=float32, chunksize=(82, 121, 99156), chunktype=numpy.ndarray>
Coordinates:
  * latitude               (latitude) float64 656B 7.75 8.0 8.25 ... 27.75 28.0
  * longitude              (longitude) float64 968B -89.0 -88.75 ... -59.0
  * fixed_time             (fixed_time) datetime64[ns] 3MB 1981-01-01T06:00:0...
    time                   (fixed_time) object 3MB MultiIndex
    forecast_initial_time  (fixed_time) datetime64[ns] 3MB 1981-01-01T06:00:0...
    forecast_hour          (fixed_time) int32 2MB 1 2 3 4 5 6 7 ... 1 2 3 4 5 6
Attributes: (12/14)
    long_name:                                          Mean vertically integ...
    short_name:                                         mvimd
    units:                                              kg m**-2 s**-1
    original_format:                                    WMO GRIB 1 with ECMWF...
    ecmwf_local_table:                                  235
    ecmwf_parameter:                                    54
    ...                                                 ...
    grid_specification:                                 0.25 degree x 0.25 de...
    rda_dataset:                                        ds633.0
    rda_dataset_url:                                    https:/rda.ucar.edu/d...
    rda_dataset_doi:                                    DOI: 10.5065/BH6N-5N20
    rda_dataset_group:                                  ERA5 atmospheric surf...
    QuantizeGranularBitGroomNumberOfSignificantDigits:  7

In [66]:
hi_ds

<xarray.Dataset> Size: 16GB
Dimensions:    (time: 394464, latitude: 82, longitude: 121)
Coordinates:
  * time       (time) datetime64[ns] 3MB 1981-01-01 ... 2025-12-31T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Data variables:
    HI_hourly  (time, latitude, longitude) float32 16GB dask.array<chunksize=(17166, 82, 121), meta=np.ndarray>

In [62]:
hi_ds_coarse = hi_ds.coarsen(time=24, boundary='trim').construct(time=('day', 'hour'))
MVIMD_fxtime_coarse = MVIMD_fxtime.coarsen(fixed_time=24, boundary='trim').construct(fixed_time=('day', 'hour'))
MSDRSWRF_fxtime_coarse = MSDRSWRF_fxtime.coarsen(fixed_time=24, boundary='trim').construct(fixed_time=('day', 'hour'))

hidmax_idx = hi_ds_coarse['HI_hourly'].argmax(dim='hour')

def sel_at_idx(ds, idx_ds):
    # add hour dimension to indexing array
    idx_ds = np.expand_dims(idx_ds, axis=-1)
    # select along hour axis and remove hour axis
    return np.take_along_axis(ds, idx_ds, axis=-1).squeeze(axis=-1)

In [63]:
hi_ds_coarse

<xarray.Dataset> Size: 16GB
Dimensions:    (day: 16436, hour: 24, latitude: 82, longitude: 121)
Coordinates:
    time       (day, hour) datetime64[ns] 3MB 1981-01-01 ... 2025-12-31T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Dimensions without coordinates: day, hour
Data variables:
    HI_hourly  (day, hour, latitude, longitude) float32 16GB dask.array<chunksize=(715, 24, 82, 121), meta=np.ndarray>

In [64]:
MVIMD_fxtime_coarse

<xarray.DataArray 'MVIMD' (latitude: 82, longitude: 121, day: 16435, hour: 24)> Size: 16GB
dask.array<reshape, shape=(82, 121, 16435, 24), dtype=float32, chunksize=(82, 121, 4131, 24), chunktype=numpy.ndarray>
Coordinates:
  * latitude               (latitude) float64 656B 7.75 8.0 8.25 ... 27.75 28.0
  * longitude              (longitude) float64 968B -89.0 -88.75 ... -59.0
    time                   (day, hour) object 3MB (Timestamp('1981-01-01 06:0...
    forecast_initial_time  (day, hour) datetime64[ns] 3MB 1981-01-01T06:00:00...
    forecast_hour          (day, hour) int32 2MB 1 2 3 4 5 6 ... 7 8 9 10 11 12
    fixed_time             (day, hour) datetime64[ns] 3MB 1981-01-01T06:00:00...
Dimensions without coordinates: day, hour
Attributes: (12/14)
    long_name:                                          Mean vertically integ...
    short_name:                                         mvimd
    units:                                              kg m**-2 s**-1
    original_format:                                    WMO GRIB 1 with ECMWF...
    ecmwf_local_table:                                  235
    ecmwf_parameter:                                    54
    ...                                                 ...
    grid_specification:                                 0.25 degree x 0.25 de...
    rda_dataset:                                        ds633.0
    rda_dataset_url:                                    https:/rda.ucar.edu/d...
    rda_dataset_doi:                                    DOI: 10.5065/BH6N-5N20
    rda_dataset_group:                                  ERA5 atmospheric surf...
    QuantizeGranularBitGroomNumberOfSignificantDigits:  7

In [65]:
MVIMD_hidmax = xr.apply_ufunc(sel_at_idx, MVIMD_fxtime_coarse, hidmax_idx,
                              input_core_dims=[['hour'], []],
                              output_core_dims=[[]],
                              dask='parallelized',
                              output_dtypes=[MVIMD_fxtime_coarse.dtype]
                              )
MVIMD_hidmax = MVIMD_hidmax.rename('MVIMD_HIdmax')
MVIMD_hidmax

AlignmentError: cannot reindex or align along dimension 'day' because of conflicting dimension sizes: {16435, 16436}